## 課題 Excel業務の自動化のアイデアを考えてみよう

### 仕様
- 見積依頼内容からLLMで製品名をメーカーの規格に沿った名称で抽出します
- メーカーの規格に沿った名称は変換対応表.xlsxに記載されており、これをFew-Shot知識として与えます
- 本来は寸法や数量といったデータも抽出する必要がありますが、ここでは製品名のみを考えます
- 実際のデータは用意できないので、実行結果は確認できていません

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
train_df = pd.read_excel("変換対応表.xlsx", index=False)
pre_prompt = f"変換対応表\n{train_df.astype(str)}\nこの変換対応表を参考にして次のテキストから製品名を抽出してください\n"

In [ ]:
def extract_product(text, pre_prompt=pre_prompt):
    """見積依頼書から製品名を抽出・メーカーの規格に変換する関数"""

    # プロンプトの作成
    prompt_text = pre_prompt + text

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": prompt_text},
            ],
            temperature=0,  # バラツキを抑える
            max_tokens=100,
        )
        # キーワード抽出結果を取得
        product_name = response.choices[0].message.content.strip()
        return product_name

    except Exception as e:
        print(f"API呼び出しでエラーが発生しました: {e}")
        return "エラー"

In [ ]:
# ワークフロー化
print("処理を開始します。")

# Excelファイルを読み込む
df = pd.read_excel('見積依頼書.xlsx', index=False)

# A列の各行のアンケートに対してキーワード抽出を実行し、B列に書き込む
df['製品名'] = df['見積依頼内容'].apply(extract_product)

# 結果をExcelファイルに保存
df.to_excel("見積依頼書_製品名変換後.xlsx", index=False)

print("Excelファイルに結果を保存しました。")